In [1]:
import pandas as pd
import openpyxl
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import re
from operator import itemgetter

In [2]:
pd.set_option('display.max_rows', None)

In [26]:
path_data = '/Users/emma/projects/cognoscolab/rassereno/data'

## Data extraction

In [27]:
raw_data_2025 = pd.read_csv(f"{path_data}/data_2025.csv")
raw_data_2026 = pd.read_csv(f"{path_data}/data_2026.csv")[2:]

In [28]:
raw_data = pd.concat([raw_data_2025, raw_data_2026], ignore_index=True)

In [29]:
# remove all the columns that are not needed and the first two rows which are 'index' of the columns
data = raw_data.drop(columns=["EndDate", "Status", "IPAddress", "RecordedDate", "ResponseId", "RecipientLastName", "RecipientFirstName", "RecipientEmail", "ExternalReference", "LocationLatitude", "LocationLongitude", "DistributionChannel", "UserLanguage"])[2:]

# convert the progress values into integers
progress = data['Progress'].values.astype(int)

# keep only the participations with more than 65% progress
data = data[(progress > 65) & (data['QID3'] == '1')].reset_index(drop=True)

## Participant ID creation

In [44]:
# ensure 'StartDate' is in datetime format
data['StartDate'] = pd.to_datetime(data['StartDate'])

# function to create participant IDs based on question 20 ('where do you study?')
def participant_id_question(q20_value, prefix):
    mask = data["Q20"] == q20_value
    data.loc[mask, 'participant_id'] = [f"{prefix}_{i+1:03d}" for i in range(mask.sum())]

# function to create participant IDs based only on specific date
def participant_id_date(dates, prefix):
    # convert single date string to list
    if isinstance(dates, str):
        dates = [dates]  

    # create a combined mask for all dates
    combined_mask = False
    for date in dates:
        mask = data['StartDate'].dt.date == pd.to_datetime(date).date()
        combined_mask |= mask  # Combine masks with OR

    if prefix == 'fisica_stem':
        data.loc[combined_mask, 'participant_id'] = [f"{prefix}_{i+1:03d}_1" for i in range(combined_mask.sum())]
    else:
        data.loc[combined_mask, 'participant_id'] = [f"{prefix}_{i+1:03d}" for i in range(combined_mask.sum())]


# function to create participant IDs based on specific time intervals
def participant_id_time(start_time, end_time, prefix, start_idx=1):
    mask = (data['StartDate'] >= start_time) & (data['StartDate'] < end_time) & (data["Q20"] == '3')
    count = mask.sum()
    data.loc[mask, 'participant_id'] = [
        f"{prefix}_{i:03d}" for i in range(start_idx, start_idx + count)
    ]
    return mask, start_idx + count

In [45]:
participant_id_question('1', 'fisica')
participant_id_question('2', 'psicologia')
participant_id_question('3', 'liceo')

participant_id_date("2025-03-26", 'interfacce')

# fisica_stem is for participants that particiapated in the gruppo di ascolto per l'ansia da STEM
participant_id_date(["2025-05-21", "2025-05-29", "2025-06-17", "2025-11-10", "2026-01-08"], 'fisica_stem')

idx_sci = 1

mask_sci, idx_sci = participant_id_time("2025-05-13 11:40:00", "2025-05-13 11:45:00", 'liceo_sci', idx_sci)
mask_lin, _ = participant_id_time("2025-05-13 12:27:00", "2025-05-13 12:40:00", 'liceo_lin', 1)
mask_sci_2, idx_sci = participant_id_time("2025-05-30 07:50:00", "2025-05-30 08:55:00", 'liceo_sci', idx_sci+1)

# create participant_id for participant in row 65 (from liceo but some days later)
data.loc[65, 'participant_id'] = f"liceo_sci_{mask_sci.sum()+1:03d}"
print(f'Participants: {len(data["participant_id"])}')

Participants: 113


In [49]:
# stem group second roudn of questionnaires completed
mask_1 = data['StartDate'].dt.date ==  pd.to_datetime("2025-07-22").date()
data.loc[mask_1, 'participant_id'] = 'fisica_stem_002_2'

mask_2 = data['StartDate'].dt.date ==  pd.to_datetime("2025-09-29").date()
data.loc[mask_2, 'participant_id'] = 'fisica_stem_001_2'

mask_3 = data['StartDate'].dt.date ==  pd.to_datetime("2025-11-19").date()
data.loc[mask_3, 'participant_id'] = 'fisica_stem_002_3'

In [72]:
data['participant_id']

0         interfacce_001
1         interfacce_002
2         interfacce_003
3         interfacce_004
4         interfacce_005
5         interfacce_006
6         interfacce_007
7         interfacce_008
8         interfacce_009
9         interfacce_010
10        interfacce_011
11        interfacce_012
12        interfacce_013
13        interfacce_014
14        interfacce_015
15        interfacce_016
16        interfacce_017
17        interfacce_018
18        interfacce_019
19        interfacce_020
20        interfacce_021
21        interfacce_022
22        interfacce_023
23            fisica_001
24            fisica_002
25            fisica_003
26            fisica_004
27            fisica_005
28         liceo_sci_001
29         liceo_sci_002
30         liceo_sci_003
31         liceo_sci_004
32         liceo_sci_005
33         liceo_sci_006
34         liceo_sci_007
35         liceo_sci_008
36         liceo_sci_009
37         liceo_sci_010
38         liceo_sci_011
39         liceo_sci_012


## Associations task

In [53]:
# these are the fileds used in the loops in qualtrics
field_1 = ["Scienza", "Equazione", "Matematica", "Programmazione", "Numeri", "Problema", "Dati"]
field_2 = ["Statistica", "Fisica", "Analisi", "Laboratorio", "Logica", "Informatica", "Esperimento"]
field_3 = ["Ipotesi", "Innovazione", "Scoperta", "Conoscenza", "Ricerca", "Sfida", "Stress"]
# Counselling é stata trasformata in Intellingenza_Artificiale dopo i primi 23 partecipanti
field_4 = ["Ansia", "Panico", "Scienziata", "Fallimento", "Imbarazzo", "Paura", "Intelligenza_Artificiale"] 
field_5 = ["Benessere", "Meditazione", "Docente", "Voto", "Compagni", "Genitori", "Regole"]
field_6 = ["Arte", "Vita", "Creatività", "Futuro", "Scuola", "Esame", "Scienziato"]

Dictonary to change keys 

{j}_QID71_{i}_{a} 

j = number of loop (7) 

i = $field (6) 

a = assosiation number (3) 

In [54]:
# make lists for the questions ID of the associations task questions and their text labels
keys_associations = []

keys_labels = []

# extract the questions ID and labels from the fields defined above
for j in range(1, 8):
    for i in range(1, 7):
        for a in range (1, 4):
            keys_associations.append(f"{j}_QID71_{i}_{a}") 
            keys_labels.append(globals()[f"field_{i}"][j - 1])

# create a dictionary to store the associations for each keyword
keywords_dict = defaultdict(list)

for key, value in data.items():
    # 126 = 42 keywords * 3 associations
    for i in range(126):
        if key == keys_associations[i]: 
            if keys_labels[i] not in keywords_dict:
                keywords_dict[keys_labels[i]] = [] 
            keywords_dict[keys_labels[i]].append(value.tolist())

series_associations_df = pd.DataFrame.from_dict(keywords_dict)

# order of the cue words that will be presented in valence task
valence_keywords_order = ['Scienza', 'Statistica', 'Ipotesi', 'Ansia', 'Benessere', 'Equazione', 'Fisica', 'Innovazione', 'Panico', 'Meditazione', 'Matematica', 'Analisi', 'Scoperta', 'Scienziata', 'Docente', 'Programmazione', 'Laboratorio', 'Conoscenza', 'Fallimento', 'Voto', 'Numeri', 'Logica', 'Ricerca', 'Imbarazzo', 'Compagni', 'Problema', 'Informatica', 'Sfida', 'Paura', 'Genitori', 'Dati', 'Esperimento', 'Stress', "Intelligenza_Artificiale", 'Regole', 'Scienziato', 'Scuola', 'Creatività', 'Arte', 'Esame', 'Futuro', 'Vita']

# reorder the rows of df_keywords according to valence_keywords_order
series_associations_df = series_associations_df[valence_keywords_order]

# transpose the DataFrame to have cue words as rows and associations as columns
df_transposed = series_associations_df.T.reset_index()

df_transposed.columns = ['cue_word', 'association_1', 'association_2', 'association_3']

In [55]:
# set as index the cue words
associations_task_df = df_transposed.set_index('cue_word')

# explode the associations into separate rows and reset the index
associations_task_df = associations_task_df.apply(pd.Series.explode).reset_index()

# add the participant_id column multiplying the participant IDs for each cue word
associations_task_df["participant_id"] = np.tile(data["participant_id"].values, len(valence_keywords_order))

associations_task_df = associations_task_df[['participant_id', 'cue_word', 'association_1', 'association_2', 'association_3']]

In [56]:
# Counselling é stata trasformata in Intellingenza_Artificiale dopo i primi 23 partecipanti
mask = associations_task_df['participant_id'].astype(str).str.startswith('interfacce') & (associations_task_df['cue_word'] == 'Intelligenza_Artificiale')

associations_task_df.loc[mask, 'cue_word'] = 'Counselling'

In [57]:
associations_task_df = associations_task_df.map(lambda x: x.lower() if isinstance(x, str) else x)

In [58]:
# to check how many cue word is repeated to see if there is one for each participant
word_counts = associations_task_df['cue_word'].value_counts().reset_index()

## Valence Task

### Structure of questions indices
j (9) = number of iteration (9 * 5(n questions) - 3 iterations)  

i (4) = number of text box (1: keyword; 2: first ass; 3: second ass; 4: third ass) 

1. first question 
{j}_Q20#{i}_1
2. second question 
{j}_QID63#{i}_1
3. third question 
{j}_QID64#{i}_1
4. fourth question 
{j}_Q15#{i}_1
5. fifth question 
{j}_Q16#{i}_1

### Extraction of association keys and values in valence task

In [59]:
# IDs of the questions of the valence task
q_ids = ["Q20", "QID63", "QID64", "Q15", "Q16"]

# creation of list to store the keys and values of the valence task
keys_valence= []
values_valence = []

# extraction of all the questions IDs (keys) and the values based on the q_id and their indices (see expalanation above)
for key, value in data.items():
    for j in range(1, 10):
        for q_id in q_ids:
          for i in range(5):
            if key == f"{j}_{q_id}#{i}_1":
              # check if the column is not empty (if there are all NaN values)
              if not data[key].iloc[:].isna().all():
                  keys_valence.append(key)
                  values_valence.append(data[key])

# create a dictionary with the keys (q_ids) and values extracted from the valence task
valence_dict = dict(zip(keys_valence, values_valence))

### Combination of dataframes of valence and association tasks

In [60]:
# create lists for questions IDs labels
labels = []

# loop for the 4 associations valence scores (valence of the cue word and of the 3 associations) and make a single dictionary

    # take each of the cue word from valence_keywords_order and create labels for each association in the valence task
for word_key in valence_keywords_order:
    for i in range(4):
        labels.append(f"{word_key}_association_{i}")

# create a dict with the list of all questions IDs (previously created) and the label associated to each of them
id_to_label = dict(zip(keys_valence, labels))

In [61]:
# assign to the question IDs the question label so that i understand to what the scores are associeted
labeled_scores = pd.DataFrame.from_dict(valence_dict, orient='index').T
labeled_scores.rename(columns=id_to_label, inplace=True)

# melt to long format: each row is one valence_key per participant
melted_valence_df = labeled_scores.melt(var_name="key", value_name="valence")

# assign an index to the df, based on the number of association (0 to 4) so that i can make separate columns for each of them
melted_valence_df["index"] = melted_valence_df["key"].str.extract(r'association_(\d)').astype(int)

# adds a new column ("row") to valence_df that tells the position of each row within its group based on the "index" column
melted_valence_df["row"] = melted_valence_df.groupby("index").cumcount()

# reshapes to wide format using as index the row as columns headers the index and as values the key and valence
df_pivoted = melted_valence_df.pivot(index="row", columns="index", values=["key", "valence"])

# create names for the columns based on their number of association and values 
df_pivoted.columns = [f"{col}_association_{i}" for col, i in df_pivoted.columns]

# reset the index without reassing it to a new variable
df_pivoted.reset_index(drop=True, inplace=True)

In [62]:
# concatenate the df with the labels for the valence task and the df of the association taks (previously created)
associations_valence_df = pd.concat([associations_task_df, df_pivoted], axis=1)

columns_order = [
    'participant_id', 
    'key_association_0',
    'cue_word',
    'valence_association_0',
    'key_association_1',
    'association_1',
    'valence_association_1',
    'key_association_2',
    'association_2',
    'valence_association_2',
    'key_association_3',
    'association_3',
    'valence_association_3'
]

# redorder the df based on columns_order and remove the columns contaning the labels of the questions IDs in the valence task
associations_valence_df = associations_valence_df[columns_order].drop(columns=['key_association_0', 'key_association_1', 'key_association_2', 'key_association_3'])

# rename the column related to the cue word in the valence task
associations_valence_df = associations_valence_df.rename(columns={'valence_association_0': 'valence_cue_word'})

### Valence df creation and exporting to csv

In [63]:
path_out = '/Users/emma/projects/cognoscolab/rassereno/out/formatting/all_students_2025'

In [77]:
# get the percentage of missing data (NaN values) for each participants
missing_values = (
    associations_valence_df
    .set_index("participant_id")
    .isna()
    .mean(axis=1)
    .groupby(level=0)
    .mean()                  
)
# make a list of participants to drop based of the percentage of missing data (more than the 1/3 of data)
high_missing = missing_values[missing_values > 1/3]

# exclude participants whose ID starts with 'fisica_stem'
to_drop = high_missing[~high_missing.index.str.startswith('fisica_stem')].index

print("Participants to drop:", len(to_drop), list(to_drop))

# create a df where we remove all the participants of the list to_drop
associations_valence = associations_valence_df[~associations_valence_df["participant_id"].isin(to_drop)].reset_index(drop=True)

# export the df to csv
associations_valence.to_csv(f'{path_out}/associations_valence_task_2025.csv', index = False)

Participants to drop: 10 ['fisica_001', 'fisica_002', 'fisica_004', 'fisica_006', 'fisica_008', 'fisica_009', 'fisica_019', 'fisica_023', 'liceo_lin_019', 'liceo_sci_015']


## Parte Questionari

### Gender

In [84]:
# extracts gender information
gender = pd.DataFrame(data['Q17']).replace({
    '1': 'M', # male
    '2': 'F', # female
    '4': 'NB', # non-binary
    '5': 'NA' # prefer not to say
})

gender.columns = ['gender']

### Competenze matematica

In [85]:
# math abilities self perception
percieved_abilities = pd.DataFrame(data['Q16'])
percieved_abilities.columns = ['percieved_abilities']

# math grade
math_grade_uni = data['Q20_1'].replace(31, '30L')
math_grade_liceo = data['Q21_1'].replace(11,'10L')

# combine the two math grade columns, prioritizing university grades over liceo grades 
math_grade = math_grade_uni.combine_first(math_grade_liceo)
math_grade_df = pd.DataFrame({'math_grade': math_grade})

### Tutti i questionari

In [86]:
# Function to extract the data of some columns corresponding to a specific administrated questionnaire
# It also renames the columns with the name of the questionnaire and number of question

def extract_block(data, question_id, start=1, end=13, col_label = 'questionnaire'):
    list_questions = []
    for i in range(start, end):
        list_questions.append(data[f'{question_id}_{i}'])

    df =  pd.DataFrame(list_questions).transpose()
    df.columns = [f'{col_label}_{i}' for i in range(start, end)]

    return df

In [87]:
# Cognitive Flexibility Scale
cfs_data = extract_block(data, 'Q22#1', 1, 13, col_label = 'cfs')

# STAI: State Anxiety
stai_data = extract_block(data, 'QID77', 1, 19, col_label = 'state_anxiety')

# Big 5: 10 items
big5_data = extract_block(data, 'QID84', 1, 11, col_label = 'big_5')

# MAS_IT reduced
mas_it_data = extract_block(data, 'Q21#1', 1, 19, col_label = 'mas_it')

mas_it_data.columns = ['mas_it_1', 'mas_it_3', 'mas_it_4', 'mas_it_5', 'mas_it_6', 'mas_it_7', 'mas_it_8', 'mas_it_9', 'mas_it_11', 'mas_it_12', 'mas_it_13', 'mas_it_14', 'mas_it_18', 'mas_it_19', 'mas_it_20', 'mas_it_21', 'mas_it_22', 'mas_it_23']

In [88]:
mas_it_df = pd.concat([data['participant_id'], mas_it_data], axis=1)
mas_it_df = mas_it_df[~mas_it_df["participant_id"].isin(to_drop)].reset_index(drop=True)
mas_it_df.to_csv(f'{path_out}/mas_it_2025.csv', index=False)

### Saving all the dfs into a csv file

In [89]:
# combines all the df and the participant_id into one and saves it to csv file
combined_dfs = pd.concat([data['participant_id'],gender, percieved_abilities, math_grade, cfs_data, stai_data, big5_data, mas_it_data], axis=1)
combined_dfs = combined_dfs[~combined_dfs["participant_id"].isin(to_drop)].reset_index(drop=True)
combined_dfs.to_csv(f'{path_out}/all_questionnaires_2025.csv', index=False)

## Free text

In [90]:
# extract the text of the free text question (Q18)
text = pd.DataFrame(data['Q18'])
text.columns = ['text']

# convert to string in lower case
text['text'] = text['text'].str.lower()

# remove trailing punctuation/whitespace, then add a full stop
text['text'] = text['text'].str.rstrip(".!? \n\t") + "."

# add participant_id
text = pd.concat([data['participant_id'], text], axis=1)

text_df = text[~text["participant_id"].isin(to_drop)].reset_index(drop=True)

text_df.to_csv(f'{path_out}/free_text.csv', index=False)